In [80]:
print("Hello World")

Hello World


In [81]:
!pip install tensorflow pandas scikit-learn

In [82]:
import tensorflow as tf

In [83]:
from tensorflow.keras.applications import DenseNet121

In [84]:
model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [85]:
model.trainable = False

In [86]:
model.summary

<bound method Model.summary of <Functional name=densenet121, built=True>>

In [87]:
from tensorflow.keras.layers import GlobalAveragePooling2D,Dense

In [88]:
x = GlobalAveragePooling2D()(model.output)
output = Dense(5,activation="softmax")(x)

In [89]:
MODEL = tf.keras.Model(inputs=model.input,outputs=output)

In [90]:
MODEL.summary

<bound method Model.summary of <Functional name=functional_4, built=True>>

In [91]:
MODEL.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=['accuracy']
)

In [92]:
MODEL.layers

[<InputLayer name=input_layer_4, built=True>,
 <ZeroPadding2D name=zero_padding2d_8, built=True>,
 <Conv2D name=conv1_conv, built=True>,
 <BatchNormalization name=conv1_bn, built=True>,
 <Activation name=conv1_relu, built=True>,
 <ZeroPadding2D name=zero_padding2d_9, built=True>,
 <MaxPooling2D name=pool1, built=True>,
 <BatchNormalization name=conv2_block1_0_bn, built=True>,
 <Activation name=conv2_block1_0_relu, built=True>,
 <Conv2D name=conv2_block1_1_conv, built=True>,
 <BatchNormalization name=conv2_block1_1_bn, built=True>,
 <Activation name=conv2_block1_1_relu, built=True>,
 <Conv2D name=conv2_block1_2_conv, built=True>,
 <Concatenate name=conv2_block1_concat, built=True>,
 <BatchNormalization name=conv2_block2_0_bn, built=True>,
 <Activation name=conv2_block2_0_relu, built=True>,
 <Conv2D name=conv2_block2_1_conv, built=True>,
 <BatchNormalization name=conv2_block2_1_bn, built=True>,
 <Activation name=conv2_block2_1_relu, built=True>,
 <Conv2D name=conv2_block2_2_conv, built=T

In [93]:
from tensorflow.keras.applications.densenet import preprocess_input

image_size=(224,224)
batch_size=32

In [94]:
train = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/train',
    image_size=image_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
    seed=42
)

Found 6054 files belonging to 5 classes.


In [95]:
valid = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/val',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 2016 files belonging to 5 classes.


In [96]:
test = tf.keras.utils.image_dataset_from_directory(
    './data/lungs-disease-dataset-4-types/versions/1/Lung Disease Dataset/test',
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False,
    label_mode="categorical"
)

Found 2025 files belonging to 5 classes.


In [97]:
print("Classes:", train.class_names)

Classes: ['Bacterial Pneumonia', 'Corona Virus Disease', 'Normal', 'Tuberculosis', 'Viral Pneumonia']


In [98]:
AUTOTUNE = tf.data.AUTOTUNE
def preprocess(image,label):
    images = tf.cast(image, tf.float32)
    images = preprocess_input(images)
    return images, label

In [99]:
train = train.map(preprocess,num_parallel_calls=AUTOTUNE)
valid = valid.map(preprocess,num_parallel_calls=AUTOTUNE)

In [100]:
train = train.prefetch(AUTOTUNE)
valid = valid.prefetch(AUTOTUNE)

In [101]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-7
    )
]

Learning has slowed down. Take smaller steps. - ReduceLROnPlateau

In [102]:
history = MODEL.fit(
    train,
    validation_data=valid,
    callbacks=callbacks,
    verbose=1,
    epochs=100
)

Epoch 1/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 155s 779ms/step - accuracy: 0.2925 - loss: 1.7019 - val_accuracy: 0.4405 - val_loss: 1.3260 - learning_rate: 1.0000e-04
Epoch 2/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 139s 729ms/step - accuracy: 0.5641 - loss: 1.1259 - val_accuracy: 0.6171 - val_loss: 1.0107 - learning_rate: 1.0000e-04
Epoch 3/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 134s 704ms/step - accuracy: 0.6842 - loss: 0.8902 - val_accuracy: 0.6959 - val_loss: 0.8451 - learning_rate: 1.0000e-04
Epoch 4/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 128s 671ms/step - accuracy: 0.7278 - loss: 0.7597 - val_accuracy: 0.7331 - val_loss: 0.7455 - learning_rate: 1.0000e-04
Epoch 5/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 128s 674ms/step - accuracy: 0.7562 - loss: 0.6795 - val_accuracy: 0.7594 - val_loss: 0.6812 - learning_rate: 1.0000e-04
Epoch 6/100
190/190 ━━━━━━━━━━━━━━━━━━━━ 127s 666ms/step - accuracy: 0.7793 - loss: 0.6232 - val_accuracy: 0.7713 - val_loss: 0.6363 - learning_rate: 1.0000e-04
Epoch 7/100
190/190 ━━━━━━━━━━━━━━

In [103]:
import joblib

In [104]:
joblib.dump(MODEL,'model.pkl')

['model.pkl']

In [106]:
import numpy as np
y_true = []
y_pred = []

for images, labels in test:
    predictions = MODEL.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)